# 🌐 Day 07: Advanced Graph Patterns — MST, Bipartite & More

---

## Minimum Spanning Tree (MST)

A spanning tree connects ALL nodes with minimum total edge weight. Two classic algorithms:

**Kruskal's:** Sort edges by weight, greedily add if it doesn't form a cycle (Union-Find!).

**Prim's:** Start from any node, always pick the cheapest edge to an unvisited node (min-heap, like Dijkstra).

```
Think of it like: building a road network connecting all cities
with minimum total road length. No loops needed.
```

---

In [ ]:
from collections import defaultdict
import heapq

class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n
    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    def union(self, x, y):
        px, py = self.find(x), self.find(y)
        if px == py: return False
        if self.rank[px] < self.rank[py]: px, py = py, px
        self.parent[py] = px
        if self.rank[px] == self.rank[py]: self.rank[px] += 1
        return True

## 🎯 Problem 1: Min Cost to Connect All Points (LC #1584)

Given points on a 2D plane, find minimum cost to connect all points (Manhattan distance).

**This is MST.** We'll use Kruskal's: generate all edges, sort, greedily add with Union-Find.

In [ ]:
def minCostConnectPoints(points):
    """
    Kruskal's MST: sort edges, add if no cycle.
    Time: O(N² log N), Space: O(N²)
    """
    n = len(points)
    # Generate all edges with Manhattan distances
    edges = []
    for i in range(n):
        for j in range(i + 1, n):
            dist = abs(points[i][0] - points[j][0]) + abs(points[i][1] - points[j][1])
            edges.append((dist, i, j))
    
    edges.sort()  # Sort by weight
    uf = UnionFind(n)
    total_cost = 0
    edges_used = 0
    
    for cost, u, v in edges:
        if uf.union(u, v):  # No cycle → add this edge
            total_cost += cost
            edges_used += 1
            if edges_used == n - 1:  # MST has exactly n-1 edges
                break
    
    return total_cost

print(minCostConnectPoints([[0,0],[2,2],[3,10],[5,2],[7,0]]))  # 20

---

## 🎯 Problem 2: Is Graph Bipartite? (LC #785)

Can you color the graph with 2 colors such that no adjacent nodes share the same color?

**Approach: BFS/DFS coloring.** Assign color 0 to start. All neighbors get color 1. Their neighbors get color 0. If we find a conflict → not bipartite.

In [ ]:
from collections import deque

def isBipartite(graph):
    """
    BFS coloring. Time: O(V + E), Space: O(V)
    """
    n = len(graph)
    color = [-1] * n  # Uncolored
    
    for start in range(n):  # Handle disconnected components
        if color[start] != -1:
            continue
        color[start] = 0
        queue = deque([start])
        while queue:
            node = queue.popleft()
            for neighbor in graph[node]:
                if color[neighbor] == -1:
                    color[neighbor] = 1 - color[node]  # Opposite color
                    queue.append(neighbor)
                elif color[neighbor] == color[node]:
                    return False  # Same color adjacent → NOT bipartite
    
    return True

print(isBipartite([[1,3],[0,2],[1,3],[0,2]]))  # True
print(isBipartite([[1,2,3],[0,2],[0,1,3],[0,2]]))  # False (odd cycle)

**When does bipartite come up?** Graph coloring, team division, checking if a graph has odd-length cycles (it does ↔ not bipartite).

---

## 🎯 Problem 3: Reconstruct Itinerary (LC #332)

Given flight tickets `[from, to]`, find the itinerary starting from "JFK" that uses all tickets in lexicographical order.

**This is an Eulerian Path** — visit every EDGE exactly once. Use Hierholzer's algorithm: DFS and add to result in reverse (post-order).

In [ ]:
def findItinerary(tickets):
    """
    Hierholzer's algorithm for Eulerian path.
    Time: O(E log E), Space: O(E)
    """
    graph = defaultdict(list)
    for src, dst in sorted(tickets, reverse=True):  # Reverse sort so pop gives lex smallest
        graph[src].append(dst)
    
    route = []
    def dfs(airport):
        while graph[airport]:
            next_dest = graph[airport].pop()
            dfs(next_dest)
        route.append(airport)  # Post-order: add when stuck (no more edges)
    
    dfs("JFK")
    return route[::-1]  # Reverse post-order = correct itinerary

print(findItinerary([["MUC","LHR"],["JFK","MUC"],["SFO","SJC"],["LHR","SFO"]]))
# ["JFK", "MUC", "LHR", "SFO", "SJC"]

**Why post-order + reverse?** DFS goes as deep as possible. When stuck (dead end), we add the airport. If we had taken a wrong branch, the dead end gets added first, and we backtrack. Reversing at the end gives the correct forward order.

---

## 🗺️ Advanced Graph Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  ADVANCED GRAPH PATTERNS                                            ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  MST (connect all, min cost):                                       ║
║    Kruskal's → sort edges + Union-Find                             ║
║    Prim's    → min-heap from any start (like Dijkstra)             ║
║                                                                      ║
║  BIPARTITE (2-colorable?):                                          ║
║    BFS/DFS coloring, conflict = not bipartite                       ║
║                                                                      ║
║  EULERIAN PATH (visit every edge once):                             ║
║    Hierholzer's: DFS + post-order + reverse                        ║
║                                                                      ║
║  GRID GRAPHS: treat cell as node, 4 neighbors as edges             ║
║  IMPLICIT GRAPHS: states as nodes, transitions as edges            ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Key Idea |
|---|---------|------------|----------|
| 1584 | Min Cost to Connect All Points | Medium | Kruskal's MST |
| 785 | Is Graph Bipartite? | Medium | BFS 2-coloring |
| 332 | Reconstruct Itinerary | Hard | Hierholzer's Eulerian path |
| 886 | Possible Bipartition | Medium | Bipartite check with groups |
| 1135 | Connecting Cities With Min Cost | Medium | MST (Kruskal's or Prim's) |
| 1168 | Optimize Water Distribution | Hard | MST with virtual node |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 MST: Kruskal's (sort + UF) or Prim's (heap). n-1 edges.      │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Bipartite = 2-colorable = no odd cycles. BFS coloring.        │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Eulerian path: Hierholzer's DFS, post-order, reverse.         │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Most graph problems reduce to BFS/DFS/Dijkstra/UnionFind.     │
└──────────────────────────────────────────────────────────────────────┘
```

---

## 🎉 Chapter 05: Graphs — COMPLETE!

You've covered: fundamentals, BFS, DFS, topological sort, Union-Find, shortest path, MST, and bipartite checking. That's the full graph toolkit for interviews.

## ➡️ Next Chapter: Dynamic Programming

The big one. Overlapping subproblems, optimal substructure, and the art of defining states.